# région proche-infrarouge (NIR)
$\rightarrow$ **expérimentations en NIR**
- extraction des indices CaT et PaT de Deneb et Vega 
- réflectance de Saturne et Neptune


## les cibles : deneb et vega
Les raies de Paschen correspondent à des transitions électroniques dans l’atome d’hydrogène : $$ n \geq 4 \rightarrow n = 3 $$

Elles se situent dans le proche infrarouge (8500–8750 Å), ce qui les rend moins sensibles à l’opacité que les raies visibles (Balmer).

Elles sont donc formées plus haut dans l’atmosphère, là où le gaz est plus dilué, moins dense, et souvent en mouvement.

Leur intensité dépend du nombre d’atomes d’hydrogène excités dans les couches externes.

Elles sont donc de bons traceurs de la structure verticale : plus la raie est forte, plus la couche est dense ou étendue.

|Série	|Domaine	|Zone de formation	|Sensibilité|
|-----|----|----|-----|
|Lyman	|UV	|Très profonde	|Température, ionisation|
|Balmer	|Visible|	Moyenne atmosphère|	Température, gravité|
|Paschen	|NIR	|Couches externes	|Expansion, densité|




L’indice **CaT** (Calcium Triplet Index) permet d’estimer la gravité de surface en se basant sur les trois raies du triplet du calcium ionisé Ca II :

CaT = EW (8498) + EW (8542) + EW (8662)




L’indice **PaT** = ∑ EW_Paschen (en A˚) sur P14 (8598 Å), P15 (8545 Å), P17 (8467 Å)


Gravité de surface (log g)
- Les naines (log g ≈ 4.5) ont des raies plus étroites et moins profondes → CaT faible.
- Les géantes (log g ≈ 2.0) ont des raies plus larges et plus profondes → CaT élevé.
- Les supergéantes (log g ≈ 0.5–1.5) ont des raies modérées, parfois affectées par la température.

 
## les données
|||
|---|---|
|OBJECT|	Deneb et Vega|
|EXPTIME2|	5 x 60 s|
|SPE_RPOW|	1414|
|BSS_VHEL|	0|
|DATE-OBS|	2025-09-19T21:13:46.114|
|BSS_SITE|	MEUSE|
|BSS_INST| MAK4/FD13 + Dados200 + 25mic + PO_UranusM|
|||






In [1]:
### A faire avec :
# NIR deneb / vega (CAT / PAT)
# saturne / neptune  / uranus

## spectro dashboard
- lancer la cellule suivante
- sur 'Colormap', bouton droit : "create new view for cell output"
- redimensionner ou déplacer l'onglet créé 'Output View' 

In [2]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
db = SpectroDashboard()
db.show()


In [6]:
from specutils import Spectrum
spec_deneb = Spectrum.read('data/plouis/_deneb_20250919_883.fits')
db.show_spectrum(spec_deneb.spectral_axis, spec_deneb.flux, label='deneb')

spec_vega = Spectrum.read('data/plouis/_vega_20250919_889.fits')
db.show_spectrum(spec_vega.spectral_axis, spec_vega.flux, label='vega')

from PIL import Image
db.show_image(Image.open('data/plouis/vega.jpg').convert('L'), 'brut')



INFO: affichage du spectre 'deneb' : 4643 pts, X:[7800.5:10613.8]


INFO: affichage du spectre 'vega' : 4643 pts, X:[7800.5:10613.8]
INFO: affichage de l'image brut : bin=1, shape=(1096, 2162), min=0, avg=82.5, max=255, stddev=20.9


In [7]:
import numpy as np
from astropy import units as u
from specutils import Spectrum, analysis
from specutils.analysis import equivalent_width


LINE_DEFINITIONS = {
#    'CaT 8498 Å': {'line': (8494.0, 8502.0), 'cont': [(8474.0, 8484.0), (8510.0, 8520.0)]},
#    'CaT 8542 Å': {'line': (8538.0, 8546.0), 'cont': [(8525.0, 8535.0), (8563.0, 8577.0)]},
#    'CaT 8662 Å': {'line': (8658.0, 8666.0), 'cont': [(8640.0, 8650.0), (8675.0, 8685.0)]},
    'Pa15 8467 Å': {'line': (8464.0, 8471.0), 'cont': [(8455.0, 8460.0), (8475.0, 8480.0)]},
    'Pa14 8502 Å': {'line': (8499.0, 8506.0), 'cont': [(8485.0, 8495.0), (8510.0, 8520.0)]},
    'Pa12 8598 Å': {'line': (8594.0, 8602.0), 'cont': [(8580.0, 8590.0), (8610.0, 8620.0)]},
    'Pa11 8665 Å': {'line': (8660.0, 8670.0), 'cont': [(8640.0, 8650.0), (8675.0, 8685.0)]},
    'Pa10 8750 Å': {'line': (8745.0, 8755.0), 'cont': [(8725.0, 8735.0), (8770.0, 8780.0)]},
}

# fonction de calcul (avec intégrale utilisant np.trapezoid au lieu de specutils)
def calculate_ew_and_uncertainty(lambda_axis, flux, line_window, cont_windows):

    cont_fluxes = [f for l_start, l_end in cont_windows 
                   for f in flux[(lambda_axis.value >= l_start) & (lambda_axis.value <= l_end)]
                  ]
    if not cont_fluxes: return 0, 0, 0    # erreur dans les intervalles des lambdas des raies

    # on retire le continuum local autour des la raie
    continuum_level = np.median(cont_fluxes)
    continuum_std = np.std(cont_fluxes)

    # on extrait le SNR local
    snr = continuum_level / continuum_std if continuum_std > 0 else 1000

    # on encadre la raie
    mask = (lambda_axis.value >= line_window[0]) & (lambda_axis.value <= line_window[1])
    if not np.any(mask): return 0, 0, snr     #  masque pas bon

    # on extrait l'EW en faisant une intégrale (np.trapezoid)
    flux_line, lambda_line = flux[mask], lambda_axis[mask].value
    ew = np.trapezoid((continuum_level - flux_line) / continuum_level, lambda_line)

    # on calcule les incertitudes sur le SNR local
    delta_lambda = np.mean(np.diff(lambda_axis))
    sigma_ew = (np.sqrt(len(lambda_line)) * delta_lambda) / snr if snr > 0 else 0
    
    return ew, sigma_ew, snr




In [8]:
# on itère pour chaque raie

for _spec in (spec_deneb, spec_vega):
    print(f"{_spec.meta['header']['OBJECT']}")
    total_ew = 0
    uncertainties_sq = []
    for line_name in LINE_DEFINITIONS:
        defs = LINE_DEFINITIONS[line_name]
        ew, sigma_ew, snr = calculate_ew_and_uncertainty(_spec.spectral_axis, _spec.flux, defs['line'], defs['cont'])
        total_ew += ew
        uncertainties_sq.append(sigma_ew.value**2)
        print(f"  - {line_name:<12s} | EW = {ew:6.3f} ± {sigma_ew:.3f}")
    
    total_uncertainty = np.sqrt(np.sum(uncertainties_sq))
    print(f"  - TOTAL ({len(LINE_DEFINITIONS)} raies) | EW = {total_ew:6.3f} ± {total_uncertainty:.3f} Å")
    print("-" * 60)



deneb
  - Pa15 8467 Å  | EW =  1.138 ± 0.028 Angstrom
  - Pa14 8502 Å  | EW =  1.378 ± 0.048 Angstrom
  - Pa12 8598 Å  | EW =  1.537 ± 0.046 Angstrom
  - Pa11 8665 Å  | EW =  1.535 ± 0.213 Angstrom
  - Pa10 8750 Å  | EW =  1.759 ± 0.021 Angstrom
  - TOTAL (5 raies) | EW =  7.346 ± 0.226 Å
------------------------------------------------------------
vega
  - Pa15 8467 Å  | EW =  0.188 ± 0.011 Angstrom
  - Pa14 8502 Å  | EW =  0.381 ± 0.029 Angstrom
  - Pa12 8598 Å  | EW =  0.834 ± 0.074 Angstrom
  - Pa11 8665 Å  | EW =  1.419 ± 0.105 Angstrom
  - Pa10 8750 Å  | EW =  1.911 ± 0.037 Angstrom
  - TOTAL (5 raies) | EW =  4.733 ± 0.137 Å
------------------------------------------------------------


## analyse 
    
ATTENTION : sur le spectre de Deneb, on voit des interférences :
- La raie H-Paschen (entourant Pa14 et Pa15) est très proche du $\text{Ca II}$ (1) et (2).
- L'absorption de la raie Pa14 ($\lambda \approx 8502\text{ }\text{A}$) est presque entièrement masquée par l'aile de la raie de $\text{Ca II}$ (1) à $8498\text{ }\text{A}$.
- Conséquence à $R=1400$ : les raies individuelles de $\text{Ca II}$ et de Paschen se chevauchent (non-résolues).
-  Calculer une Largeur Équivalente (EW) sur Pa14 (ou même Pa13) revient à inclure de manière significative le flux d'une raie de $\text{Ca II}$ forte, ce qui surestime la force de l'hydrogène et fausse la détermination du $\log g$.


--> on ne peut pas interpréter l'indice PAT ni CAT avec ces spectres ...
